# 02 - Calidad, limpieza y preparacion



## 1. Carga y funciones auxiliares

Se carga el JSON original y se prepara una copia llamada `df`. 

In [8]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path("..").resolve()
raw_path = ROOT / "data" / "raw" / "streaming_users_dirty.json"
processed_csv_path = ROOT / "data" / "processed" / "streaming_users_processed.csv"
processed_json_path = ROOT / "data" / "processed" / "streaming_users_processed.json"
log_path = ROOT / "logs" / "pipeline_log.csv"

FECHA_CORTE = pd.Timestamp("2026-06-28")
EDAD_MIN_PLAUSIBLE = 13
EDAD_MAX_PLAUSIBLE = 100


In [9]:
def normalizar_texto(valor):
    """Quita espacios y pasa texto a minusculas para poder mapear variantes."""
    if pd.isna(valor):
        return np.nan
    return str(valor).strip().lower()


def parsear_fechas_login(serie):
    """Intenta leer fechas en formatos frecuentes y convierte errores a NaN."""
    parsed = pd.to_datetime(serie, errors="coerce", format="%Y-%m-%d")
    faltan = parsed.isna() & serie.notna()
    parsed.loc[faltan] = pd.to_datetime(serie.loc[faltan], errors="coerce", dayfirst=True)
    faltan = parsed.isna() & serie.notna()
    parsed.loc[faltan] = pd.to_datetime(serie.loc[faltan], errors="coerce")
    return parsed


def ordenar_usuarios_repetidos(base):
    """Devuelve indices ordenados para conservar la mejor fila de cada user_id."""
    login_tmp = parsear_fechas_login(base["last_login_date"])
    login_valido_tmp = login_tmp.notna() & (login_tmp <= FECHA_CORTE)
    consumo_valido_tmp = base["monthly_watch_time_mins"].between(0, 14400, inclusive="both")
    consumo_tipico = base.loc[consumo_valido_tmp, "monthly_watch_time_mins"].median()
    distancia_consumo = (base["monthly_watch_time_mins"] - consumo_tipico).abs()
    distancia_consumo = distancia_consumo.where(consumo_valido_tmp, np.inf)
    completitud_tmp = base.notna().sum(axis=1)

    ranking = pd.DataFrame({
        "_idx": base.index,
        "_login_valido": login_valido_tmp.astype(int),
        "_consumo_valido": consumo_valido_tmp.astype(int),
        "_distancia_consumo": distancia_consumo,
        "_login": login_tmp,
        "_complete": completitud_tmp,
    })

    return ranking.sort_values(
        ["_login_valido", "_consumo_valido", "_distancia_consumo", "_login", "_complete", "_idx"],
        ascending=[False, False, True, False, False, True],
    )["_idx"]


raw = pd.read_json(raw_path)
df = raw.copy()
filas_iniciales = len(df)
log = []


def registrar(paso, decision, evidencia):
    """Registra el impacto de cada etapa de limpieza."""
    log.append({
        "Paso": paso,
        "Decision": decision,
        "Evidencia": evidencia,
        "Filas": len(df),
        "Nulos": int(df.isna().sum().sum()),
        "Retencion (%)": round(len(df) / filas_iniciales * 100, 2),
    })


registrar("00", "Carga del dataset original en una copia de trabajo.", "Se preserva data/raw sin modificar.")
raw.head()

,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets
0,10000,39,Estándar,805.8,Brasil,Crime,2025-03-04,99
1,10001,37,Estándar,1173.4,Colombia,Crime,2019-04-02,2
2,10002,28,Básico,401.0,Colombia,Crime,2018-04-13,0
3,10003,43,Básico,62.4,Uruguay,Thriller,2021-01-31,0
4,10004,51,Básico,477.8,Perú,Thriller,2020-09-30,1


## 2. Diagnostico inicial de calidad

Antes de limpiar, se mide el problema. Se revisan filas, columnas, nulos, duplicados exactos, `user_id` repetidos, categorias con variantes y valores numericos fuera de rango. No alcanza con contar faltantes: tambien importa cuanta parte de la base representan y si ese volumen puede afectar el analisis.

Resumen del diagnostico inicial sobre 8160 filas:

- Nulos totales: 753 celdas, equivalente a 1.15% de las celdas del dataset.
- Duplicados exactos: 126 filas, equivalente a 1.54% de la base.
- `user_id` repetidos: 160 filas excedentes, equivalente a 1.96% de la base.
- Valores negativos en consumo: 49 filas.
- Tickets negativos: 29 filas.
- Edades fuera de rango plausible: 120 filas.

En variables de conteo como `customer_support_tickets`, los umbrales se apoyan en estadistica descriptiva (IQR y percentiles) para evitar cortes arbitrarios.

In [10]:
diagnostico_inicial = pd.DataFrame({
    "filas": [raw.shape[0]],
    "columnas": [raw.shape[1]],
    "nulos_totales": [int(raw.isna().sum().sum())],
    "user_id_repetidos": [int(raw.duplicated("user_id").sum())],
    "duplicados_exactos": [int(raw.duplicated().sum())],
})
diagnostico_inicial

,filas,columnas,nulos_totales,user_id_repetidos,duplicados_exactos
0,8160,8,753,160,126


In [11]:
raw.isna().sum().to_frame("nulos_por_columna")

,nulos_por_columna
user_id,0
age,0
subscription_plan,0
monthly_watch_time_mins,193
country,0
favorite_genre,240
last_login_date,320
customer_support_tickets,0


In [12]:
for col in ["subscription_plan", "country", "favorite_genre"]:
    print(f"\n{col}")
    print(raw[col].value_counts(dropna=False).head(25))


subscription_plan
subscription_plan
Básico       3450
Estándar     2711
Premium      1519
basico         60
BASICO         52
Basic          52
básico         50
Std            48
Estándar       46
estandar       36
STANDARD       34
Premium        31
PREMIUM        26
Premiun        23
premium        22
Name: count, dtype: int64

country
country
Brasil       1132
Chile        1132
México       1129
Uruguay      1124
Perú         1120
Colombia     1116
Argentina    1087
colombia       27
méxico         25
uruguay        24
Brazil         21
COL            19
CHL            18
URY            17
MEX            16
Chile          16
argentina      16
PER            16
chile          15
Mexico         15
Peru           15
BRA            15
brasil         13
perú           12
ARG            10
Name: count, dtype: int64

favorite_genre
favorite_genre
Comedia        1112
Drama          1105
Documental     1095
Thriller       1090
Romance        1090
Acción         1082
Crime          1067
NaN

In [13]:
edad_q1 = raw["age"].quantile(0.25)
edad_q3 = raw["age"].quantile(0.75)
edad_iqr = edad_q3 - edad_q1
edad_umbral_iqr_sup = edad_q3 + 1.5 * edad_iqr

# Regla del IQR para detectar valores altos sospechosos en tickets
# Q1: percentil 25, Q3: percentil 75, IQR = Q3 - Q1
# Umbral superior = Q3 + 1.5 * IQR
tickets_q1 = raw["customer_support_tickets"].quantile(0.25)
tickets_q3 = raw["customer_support_tickets"].quantile(0.75)
tickets_iqr = tickets_q3 - tickets_q1
tickets_umbral_sospechoso = tickets_q3 + (1.5 * tickets_iqr)

rangos_sospechosos = pd.DataFrame({
    "edad_menor_13": [(raw["age"] < EDAD_MIN_PLAUSIBLE).sum()],
    "edad_mayor_100": [(raw["age"] > EDAD_MAX_PLAUSIBLE).sum()],
    "consumo_negativo": [(raw["monthly_watch_time_mins"] < 0).sum()],
    "tickets_negativos": [(raw["customer_support_tickets"] < 0).sum()],
    "tickets_sobre_umbral_iqr": [(raw["customer_support_tickets"] > tickets_umbral_sospechoso).sum()],
})
pd.DataFrame({
    "edad_q1": [edad_q1],
    "edad_q3": [edad_q3],
    "edad_iqr": [edad_iqr],
    "edad_umbral_iqr_sup": [edad_umbral_iqr_sup],
    "edad_min_plausible": [EDAD_MIN_PLAUSIBLE],
    "edad_max_plausible": [EDAD_MAX_PLAUSIBLE],
    "q1": [tickets_q1],
    "q3": [tickets_q3],
    "iqr": [tickets_iqr],
    "umbral_iqr": [tickets_umbral_sospechoso],
})
rangos_sospechosos

,edad_menor_13,edad_mayor_100,consumo_negativo,tickets_negativos,tickets_sobre_umbral_iqr
0,67,53,49,29,439


## 3. Duplicados exactos

Primero se muestran los duplicados exactos. Estas filas no agregan informacion nueva: son registros repetidos linea por linea. Por eso se eliminan con `drop_duplicates()`.

La limpieza no se hace a ciegas: antes se mira cuantos hay y se pueden inspeccionar ejemplos.

In [14]:
duplicados_exactos = raw[raw.duplicated(keep=False)].sort_values(list(raw.columns))
print("Filas involucradas en duplicados exactos:", len(duplicados_exactos))
duplicados_exactos.head(10)

Filas involucradas en duplicados exactos: 252


,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets
37,10037,33,Básico,611.0,Colombia,Documental,2019-09-29,2
8133,10037,33,Básico,611.0,Colombia,Documental,2019-09-29,2
52,10052,25,Básico,465.7,Colombia,Acción,2022-03-31,1
8089,10052,25,Básico,465.7,Colombia,Acción,2022-03-31,1
117,10117,29,Estándar,713.9,Brasil,Documental,2020-12-20,1
8010,10117,29,Estándar,713.9,Brasil,Documental,2020-12-20,1
128,10128,19,Básico,638.7,Argentina,Drama,2020-06-17,1
8085,10128,19,Básico,638.7,Argentina,Drama,2020-06-17,1
156,10156,43,Básico,592.8,Brasil,Romance,2021-10-25,0
8000,10156,43,Básico,592.8,Brasil,Romance,2021-10-25,0


In [15]:
antes = len(df)
df = df.drop_duplicates().reset_index(drop=True)
registrar(
    "01",
    "Eliminacion de duplicados exactos.",
    f"Se eliminaron {antes - len(df)} filas repetidas linea por linea.",
)
pd.DataFrame(log).tail(1)

,Paso,Decision,Evidencia,Filas,Nulos,Retencion (%)
1,01,Eliminacion de duplicados exactos.,Se eliminaron 126 filas repetidas linea por li...,8034,753,98.46


## 4. `user_id` repetidos

Despues de eliminar duplicados exactos, todavia pueden quedar varios registros con el mismo `user_id`. En este proyecto no se imputan los `user_id` repetidos, porque un identificador no es un valor faltante: es una clave que debe resolverse eligiendo una sola fila confiable por usuario.

Ese bloque representa 160 filas excedentes, es decir, aproximadamente 1.96% del dataset original. No es un volumen enorme, pero tampoco es despreciable: si se lo dejara sin tratar, habria usuarios duplicados que inflarian conteos, distorsionarian distribuciones y podrian sesgar cualquier comparacion por perfil.

Criterio usado para elegir la mejor fila:

1. Fecha de login valida y no futura.
2. Consumo mensual plausible, entre 0 y 14400 minutos.
3. Consumo mas cercano al consumo tipico de la base.
4. Login mas reciente.
5. Mayor completitud de campos.

Las columnas auxiliares solo se usan para decidir; no se guardan en el dataset final.

In [16]:
conteo_user_id = df["user_id"].value_counts()
user_id_repetidos = conteo_user_id[conteo_user_id > 1]
print("Cantidad de user_id con mas de un registro:", len(user_id_repetidos))
print("Filas excedentes por user_id repetido:", int(df.duplicated("user_id").sum()))

muestra_ids = user_id_repetidos.head(5).index
df[df["user_id"].isin(muestra_ids)].sort_values(["user_id", "last_login_date"]).head(20)

Cantidad de user_id con mas de un registro: 34
Filas excedentes por user_id repetido: 34


,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets
59,10059,39,Estándar,2976.6,Brasil,DRAMA,2024-06-22,1
8022,10059,39,Estándar,824.8,Brasil,Drama,2024-06-22,1
8026,10721,32,Estándar,959.0,Colombia,Documental,2021-09-02,2
721,10721,32,Estándar,959.0,Colombia,Documental,NaN,2
797,10797,31,Básico,-1.0,México,Comedia,2023-07-01,1
8005,10797,31,Básico,410.4,México,Comedia,2023-07-01,1
1092,11092,31,Estándar,959.6,Chile,Romance,2025-01-05,2
8017,11092,31,Estándar,959.6,Chile,Romance,2025-01-05,2
1222,11222,13,Estándar,1321.8,MEX,Documental,2019-02-08,0
8032,11222,13,Estándar,1321.8,México,Documental,2019-02-08,0


In [17]:
antes = len(df)
orden = ordenar_usuarios_repetidos(df)
df = (
    df.loc[orden]
    .drop_duplicates(subset="user_id", keep="first")
    .sort_values("user_id")
    .reset_index(drop=True)
)
registrar(
    "02",
    "Resolucion de user_id repetidos con ranking de calidad.",
    f"Se quitaron {antes - len(df)} filas excedentes conservando el registro mas confiable por usuario.",
)
pd.DataFrame(log).tail(1)

C:\Users\Fernando Sanchez\AppData\Local\Temp\ipykernel_8008\2193693530.py:12: UserWarning: Parsing dates in %m-%d-%Y format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  parsed.loc[faltan] = pd.to_datetime(serie.loc[faltan], errors="coerce", dayfirst=True)
C:\Users\Fernando Sanchez\AppData\Local\Temp\ipykernel_8008\2193693530.py:14: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed.loc[faltan] = pd.to_datetime(serie.loc[faltan], errors="coerce")


,Paso,Decision,Evidencia,Filas,Nulos,Retencion (%)
2,02,Resolucion de user_id repetidos con ranking de...,Se quitaron 34 filas excedentes conservando el...,8000,743,98.04


## 5. Categorias inconsistentes

Las variables categoricas tenian variantes equivalentes. Por ejemplo, `std`, `standard` y `estandar` representan el mismo plan. Si no se normalizan, el analisis cuenta grupos falsos.

Primero se miran las categorias, luego se aplica un mapa de equivalencias.

In [18]:
categorias_antes = {
    col: sorted(df[col].dropna().astype(str).unique().tolist())
    for col in ["subscription_plan", "country", "favorite_genre"]
}
categorias_antes

{'subscription_plan': ['BASICO',
  'Basic',
  'Básico',
  'Estándar',
  'Estándar ',
  'PREMIUM',
  'Premium',
  'Premium ',
  'Premiun',
  'STANDARD',
  'Std',
  'basico',
  'básico',
  'estandar',
  'premium'],
 'country': ['ARG',
  'Argentina',
  'Argentina ',
  'BRA',
  'Brasil',
  'Brazil',
  'CHL',
  'COL',
  'Chile',
  'Chile ',
  'Colombia',
  'MEX',
  'Mexico',
  'México',
  'PER',
  'Peru',
  'Perú',
  'URY',
  'Uruguay',
  'argentina',
  'brasil',
  'chile',
  'colombia',
  'méxico',
  'perú',
  'uruguay'],
 'favorite_genre': ['ACCIÓN',
  'Acción',
  'Action',
  'COMEDIA',
  'CRIME',
  'Comedia',
  'Comedia ',
  'Crime',
  'Crimen',
  'DOC',
  'DRAMA',
  'Documental',
  'Documentary',
  'Drama',
  'Drama ',
  'ROMANCE',
  'Romance',
  'Romance ',
  'THRILLER',
  'Thriller',
  'Thriller ',
  'accion',
  'comedy',
  'crime',
  'documental',
  'drama',
  'romance',
  'thriler']}

In [19]:
mapa_plan = {
    "estandar": "Estandar", "est?ndar": "Estandar", "std": "Estandar", "standard": "Estandar",
    "basico": "Basico", "b?sico": "Basico", "basic": "Basico",
    "premium": "Premium", "premiun": "Premium",
}
mapa_pais = {
    "argentina": "Argentina", "arg": "Argentina",
    "brasil": "Brasil", "brazil": "Brasil", "bra": "Brasil",
    "chile": "Chile", "chl": "Chile",
    "colombia": "Colombia", "col": "Colombia",
    "mexico": "Mexico", "m?xico": "Mexico", "mex": "Mexico",
    "peru": "Peru", "per?": "Peru", "per": "Peru",
    "uruguay": "Uruguay", "ury": "Uruguay",
}
mapa_genero = {
    "accion": "Accion", "acci?n": "Accion", "action": "Accion",
    "comedia": "Comedia", "comedy": "Comedia",
    "crime": "Crimen", "crimen": "Crimen",
    "documental": "Documental", "documentary": "Documental", "doc": "Documental",
    "drama": "Drama", "romance": "Romance",
    "thriller": "Thriller", "thriler": "Thriller",
}

df["subscription_plan"] = df["subscription_plan"].map(lambda x: mapa_plan.get(normalizar_texto(x), x))
df["country"] = df["country"].map(lambda x: mapa_pais.get(normalizar_texto(x), x))
df["favorite_genre"] = df["favorite_genre"].map(lambda x: mapa_genero.get(normalizar_texto(x), np.nan if pd.isna(x) else x))

registrar(
    "03",
    "Estandarizacion de categorias equivalentes.",
    "Se unificaron variantes de plan, pais y genero favorito para evitar grupos duplicados por escritura.",
)

categorias_despues = {
    col: sorted(df[col].dropna().astype(str).unique().tolist())
    for col in ["subscription_plan", "country", "favorite_genre"]
}
categorias_despues

{'subscription_plan': ['Basico',
  'Básico',
  'Estandar',
  'Estándar',
  'Estándar ',
  'Premium',
  'básico'],
 'country': ['Argentina',
  'Brasil',
  'Chile',
  'Colombia',
  'Mexico',
  'México',
  'Peru',
  'Perú',
  'Uruguay',
  'méxico',
  'perú'],
 'favorite_genre': ['ACCIÓN',
  'Accion',
  'Acción',
  'Comedia',
  'Crimen',
  'Documental',
  'Drama',
  'Romance',
  'Thriller']}

## 6. Valores imposibles o no plausibles

En esta etapa no se imputan directamente los errores. Primero se convierten a nulos para diferenciarlos de valores validos.

Reglas aplicadas:

- Edad menor a 13 o mayor a 100: no plausible para este caso.
- Tiempo mensual negativo: imposible.
- Tickets negativos: imposible.
- Fechas invalidas o posteriores a la fecha de corte: no confiables.

In [20]:
problemas_antes = pd.DataFrame({
    "edad_fuera_13_100": [((df["age"] < 13) | (df["age"] > 100)).sum()],
    "consumo_negativo": [(df["monthly_watch_time_mins"] < 0).sum()],
    "tickets_negativos": [(df["customer_support_tickets"] < 0).sum()],
    "fechas_invalidas_o_futuras": [((parsear_fechas_login(df["last_login_date"]).isna()) | (parsear_fechas_login(df["last_login_date"]) > FECHA_CORTE)).sum()],
})
problemas_antes

C:\Users\Fernando Sanchez\AppData\Local\Temp\ipykernel_8008\2193693530.py:12: UserWarning: Parsing dates in %m-%d-%Y format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  parsed.loc[faltan] = pd.to_datetime(serie.loc[faltan], errors="coerce", dayfirst=True)
C:\Users\Fernando Sanchez\AppData\Local\Temp\ipykernel_8008\2193693530.py:14: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed.loc[faltan] = pd.to_datetime(serie.loc[faltan], errors="coerce")
C:\Users\Fernando Sanchez\AppData\Local\Temp\ipykernel_8008\2193693530.py:12: UserWarning: Parsing dates in %m-%d-%Y format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  parsed.loc[faltan] = pd.to_datetime(serie.loc[faltan], errors="coerce", dayfirst=True)
C:\Users\Fernando Sanchez\AppData\Lo

,edad_fuera_13_100,consumo_negativo,tickets_negativos,fechas_invalidas_o_futuras
0,120,48,29,394


In [21]:
df.loc[(df["age"] < EDAD_MIN_PLAUSIBLE) | (df["age"] > EDAD_MAX_PLAUSIBLE), "age"] = np.nan
df.loc[df["monthly_watch_time_mins"] < 0, "monthly_watch_time_mins"] = np.nan
df.loc[df["customer_support_tickets"] < 0, "customer_support_tickets"] = np.nan

login = parsear_fechas_login(df["last_login_date"])
login.loc[login > FECHA_CORTE] = pd.NaT
df["last_login_date"] = login

registrar(
    "04",
    "Conversion de valores imposibles a nulos.",
    f"Se marcaron como nulos edades fuera de {EDAD_MIN_PLAUSIBLE}-{EDAD_MAX_PLAUSIBLE}, consumos negativos, tickets negativos y fechas invalidas/futuras.",
)
df.isna().sum().to_frame("nulos_despues_de_marcar_errores")

C:\Users\Fernando Sanchez\AppData\Local\Temp\ipykernel_8008\2193693530.py:12: UserWarning: Parsing dates in %m-%d-%Y format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  parsed.loc[faltan] = pd.to_datetime(serie.loc[faltan], errors="coerce", dayfirst=True)
C:\Users\Fernando Sanchez\AppData\Local\Temp\ipykernel_8008\2193693530.py:14: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed.loc[faltan] = pd.to_datetime(serie.loc[faltan], errors="coerce")


,nulos_despues_de_marcar_errores
user_id,0
age,120
subscription_plan,0
monthly_watch_time_mins,239
country,0
favorite_genre,237
last_login_date,394
customer_support_tickets,29


## 6.1. Mecanismo de faltantes

Antes de imputar, conviene pensar por que faltan los datos. Esa distincion ayuda a decidir si conviene imputar, eliminar o dejar el valor ausente.

- **MCAR**: la ausencia no depende de nada observado ni no observado. Borrar filas no introduce sesgo, pero es un caso poco frecuente.
- **MAR**: la ausencia depende de variables observadas. En ese caso, la imputacion por segmento o condicionada en variables disponibles es razonable.
- **MNAR**: la ausencia depende del propio valor faltante. Es el caso mas delicado; imputar sin modelar ese mecanismo puede introducir sesgo.

En este proyecto, los faltantes de edad, consumo, genero favorito y ultima fecha de login se tratan como un escenario mas cercano a **MAR** que a MCAR: la ausencia no se interpreta como aleatoria pura, sino condicionada por el contexto observado del registro (plan, pais, consistencia de fechas y rangos plausibles). Por eso se usa imputacion por segmento y mediana/moda, en lugar de eliminar filas de forma indiscriminada.

## 7. Imputacion

La imputacion reemplaza nulos con valores razonables. Se evita usar un unico promedio global cuando existe informacion de segmento.

Desde la teoria de faltantes, el criterio utilizado se interpreta mejor como un escenario **MAR** (Missing At Random) que como MCAR: la ausencia se maneja condicionandola a variables observadas como `subscription_plan` y `country`, en lugar de asumir que falta al azar puro.

Criterio usado:

- `age` y `monthly_watch_time_mins`: mediana por `subscription_plan` y `country`; si falta, mediana global.
- `customer_support_tickets`: mediana global, porque es conteo discreto.
- `favorite_genre`: moda por `subscription_plan` y `country`; si falta, moda global.
- `last_login_date`: mediana global de fechas validas.

Se usa mediana porque resiste mejor los extremos que el promedio.

In [22]:
nulos_antes_imputar = df.isna().sum().to_frame("nulos_antes_imputar")
nulos_antes_imputar

,nulos_antes_imputar
user_id,0
age,120
subscription_plan,0
monthly_watch_time_mins,239
country,0
favorite_genre,237
last_login_date,394
customer_support_tickets,29


In [23]:
for col in ["age", "monthly_watch_time_mins"]:
    df[col] = df.groupby(["subscription_plan", "country"], observed=True)[col].transform(lambda s: s.fillna(s.median()))
    df[col] = df[col].fillna(df[col].median())

df["customer_support_tickets"] = df["customer_support_tickets"].fillna(df["customer_support_tickets"].median())

df["favorite_genre"] = df.groupby(["subscription_plan", "country"], observed=True)["favorite_genre"].transform(
    lambda s: s.fillna(s.mode().iloc[0] if not s.mode().empty else np.nan)
)
df["favorite_genre"] = df["favorite_genre"].fillna(df["favorite_genre"].mode().iloc[0])

df["last_login_date"] = df["last_login_date"].fillna(df["last_login_date"].dropna().median())

registrar(
    "05",
    "Imputacion de nulos con medianas, modas y fecha mediana.",
    "La base queda sin nulos y los reemplazos se apoyan en segmentos cuando es posible.",
)
df.isna().sum().to_frame("nulos_despues_imputar")

,nulos_despues_imputar
user_id,0
age,0
subscription_plan,0
monthly_watch_time_mins,0
country,0
favorite_genre,0
last_login_date,0
customer_support_tickets,0


## 8. Valores extremos y winsorizacion

Algunos valores positivos no son imposibles, pero pueden ser demasiado extremos para el analisis. En vez de eliminar usuarios, se capean los valores superiores.

La winsorizacion conserva la fila, pero limita el efecto de extremos sobre medias, graficos, correlaciones y PCA.

In [24]:
resumen_extremos_antes = df[["monthly_watch_time_mins", "customer_support_tickets"]].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).T
resumen_extremos_antes

,count,mean,std,min,50%,75%,90%,95%,99%,max
monthly_watch_time_mins,8000.0,1103.336812,5181.538527,0.0,772.2,1062.3,1268.82,1405.21,3628.429,99999.0
customer_support_tickets,8000.0,1.827625,11.445121,0.0,1.0,1.0,2.00,3.00,5.000,150.0


In [25]:
cap_watch = df["monthly_watch_time_mins"].quantile(0.99)
cap_tickets = df["customer_support_tickets"].quantile(0.99)

valores_a_capear = pd.DataFrame({
    "variable": ["monthly_watch_time_mins", "customer_support_tickets"],
    "cap": [cap_watch, cap_tickets],
    "percentil_usado": [0.99, 0.99],
    "valores_por_encima_del_cap": [
        int((df["monthly_watch_time_mins"] > cap_watch).sum()),
        int((df["customer_support_tickets"] > cap_tickets).sum()),
    ],
})
valores_a_capear

,variable,cap,percentil_usado,valores_por_encima_del_cap
0,monthly_watch_time_mins,3628.429,0.99,80
1,customer_support_tickets,5.000,0.99,67


In [26]:
df["monthly_watch_time_mins"] = df["monthly_watch_time_mins"].clip(upper=cap_watch)
df["customer_support_tickets"] = df["customer_support_tickets"].clip(upper=cap_tickets)

registrar(
    "06",
    "Winsorizacion superior de consumo mensual y tickets.",
    f"Se aplico cap en percentil 99: watch_time={cap_watch:.1f}, tickets={cap_tickets:.0f}.",
)
df[["monthly_watch_time_mins", "customer_support_tickets"]].describe().T

,count,mean,std,min,25%,50%,75%,max
monthly_watch_time_mins,8000.0,808.938778,506.844462,0.0,498.875,772.2,1062.3,3628.429
customer_support_tickets,8000.0,0.836375,0.972224,0.0,0.000,1.0,1.0,5.000


## 9. Tipos finales y exportacion

Se normalizan tipos y se respeta la estructura original: el dataset procesado conserva las mismas columnas que el JSON inicial. Las columnas auxiliares solo se usaron dentro del proceso.

In [27]:
df["age"] = df["age"].round().astype(int)
df["customer_support_tickets"] = df["customer_support_tickets"].round().astype(int)
df["monthly_watch_time_mins"] = df["monthly_watch_time_mins"].round(1)
df["last_login_date"] = pd.to_datetime(df["last_login_date"]).dt.strftime("%Y-%m-%d")
df = df[raw.columns]

registrar(
    "07",
    "Normalizacion final de tipos y exportacion.",
    "Se exporta el dataset procesado con las mismas columnas originales.",
)

df.to_csv(processed_csv_path, index=False, encoding="utf-8")
df.to_json(processed_json_path, orient="records", force_ascii=False, indent=2)
log_df = pd.DataFrame(log)
log_df.to_csv(log_path, index=False, encoding="utf-8")

control_final = pd.DataFrame({
    "filas_finales": [len(df)],
    "columnas_finales": [df.shape[1]],
    "nulos_finales": [int(df.isna().sum().sum())],
    "duplicados_exactos_finales": [int(df.duplicated().sum())],
    "user_id_repetidos_finales": [int(df.duplicated("user_id").sum())],
    "mismas_columnas_originales": [list(df.columns) == list(raw.columns)],
})
control_final

,filas_finales,columnas_finales,nulos_finales,duplicados_exactos_finales,user_id_repetidos_finales,mismas_columnas_originales
0,8000,8,0,0,0,True


In [28]:
log_df

,Paso,Decision,Evidencia,Filas,Nulos,Retencion (%)
0,00,Carga del dataset original en una copia de tra...,Se preserva data/raw sin modificar.,8160,753,100.00
1,01,Eliminacion de duplicados exactos.,Se eliminaron 126 filas repetidas linea por li...,8034,753,98.46
2,02,Resolucion de user_id repetidos con ranking de...,Se quitaron 34 filas excedentes conservando el...,8000,743,98.04
3,03,Estandarizacion de categorias equivalentes.,"Se unificaron variantes de plan, pais y genero...",8000,743,98.04
4,04,Conversion de valores imposibles a nulos.,"Se marcaron como nulos edades fuera de 13-100,...",8000,1019,98.04
5,05,"Imputacion de nulos con medianas, modas y fech...",La base queda sin nulos y los reemplazos se ap...,8000,0,98.04
6,06,Winsorizacion superior de consumo mensual y ti...,Se aplico cap en percentil 99: watch_time=3628...,8000,0,98.04
7,07,Normalizacion final de tipos y exportacion.,Se exporta el dataset procesado con las mismas...,8000,0,98.04


## 10. Lectura final

La limpieza no solo deja una base prolija; deja una base defendible. Cada decision tiene evidencia previa, codigo aplicado e impacto medido.

Resultado final:

- Se conserva el archivo original sin tocar.
- Se eliminan duplicados exactos y usuarios repetidos con una regla explicita.
- Se unifican categorias equivalentes.
- Se tratan valores imposibles como nulos antes de imputar.
- Se imputan faltantes con criterios simples y explicables, interpretando el mecanismo como MAR cuando hay variables observadas que ayudan a explicar la ausencia.
- Se controlan extremos con umbrales calculados por IQR y percentiles, sin usar cortes al tanteo.
- Se exporta una base final sin nulos, sin duplicados y con las mismas columnas originales.
- El volumen de usuarios repetidos y nulos iniciales era suficiente para afectar conteos, porcentajes y comparaciones por perfil; por eso la limpieza s? fue significativa para el an?lisis.